# Global Places of Interest — Surf & Weather Forecast

Fetches live marine and weather data for any location using the free **open-meteo** APIs (no key needed) plus optional **surf-forecast.com** scrapes, then synthesises a per-place briefing via **Ollama (llama3.2)** running locally.

To add a new place, just add an entry to `PLACES` in the config cell.

In [1]:
import requests
from IPython.display import Markdown, display
from openai import OpenAI
from playwright.async_api import async_playwright
import json

In [2]:
try:
    r = requests.get("http://localhost:11434", timeout=3)
    print(r.text)
except Exception:
    print("Ollama is NOT running — open a terminal and run: ollama serve")

Ollama is running


In [3]:
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

## Places Config

Add as many places as you like. Each entry needs:
- `category` — `surf`, `mountain`, or `general`
- `lat` / `lon` — for open-meteo marine + wind data
- `scrape_sources` — optional dict of name → URL for Playwright scraping

In [4]:
PLACES = {
    'Taghazout — Surf': {
        'category': 'surf',
        'lat': 30.50,
        'lon': -9.70,
        'scrape_sources': {
            'Anchor Point (surf-forecast.com)': 'https://www.surf-forecast.com/breaks/Anchor-Point/forecasts/latest',
        },
    },
    # Add more places here, e.g.:
    # 'Hossegor — Surf': {
    #     'category': 'surf',
    #     'lat': 43.67,
    #     'lon': -1.44,
    #     'scrape_sources': {
    #         'La Graviere (surf-forecast.com)': 'https://www.surf-forecast.com/breaks/La-Graviere/forecasts/latest',
    #     },
    # },
}

## Data Fetchers

### open-meteo
Free, no API key, global coverage. We hit two endpoints:
- **Marine API** — swell height, period, wave direction
- **Forecast API** — wind speed and direction at 10 m

In [5]:
MARINE_PARAMS = [
    'wave_height', 'wave_period', 'wave_direction',
    'swell_wave_height', 'swell_wave_period', 'swell_wave_direction',
    'wind_wave_height',
]

WIND_PARAMS = ['wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m']


def fetch_marine(lat, lon, days=5):
    url = 'https://marine-api.open-meteo.com/v1/marine'
    params = {
        'latitude': lat, 'longitude': lon,
        'hourly': ','.join(MARINE_PARAMS),
        'forecast_days': days,
    }
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    data = r.json()['hourly']
    lines = ['open-meteo Marine Forecast (hourly, next 5 days)\n']
    for i, t in enumerate(data['time'][:120:3]):  # every 3 hours
        wh  = data['wave_height'][i*3]
        sh  = data['swell_wave_height'][i*3]
        sp  = data['swell_wave_period'][i*3]
        sd  = data['swell_wave_direction'][i*3]
        lines.append(
            f'{t}  wave:{wh}m  swell:{sh}m @ {sp}s from {sd}deg'
        )
    return '\n'.join(lines)


def fetch_wind(lat, lon, days=5):
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': lat, 'longitude': lon,
        'hourly': ','.join(WIND_PARAMS),
        'forecast_days': days,
        'wind_speed_unit': 'kn',
    }
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    data = r.json()['hourly']
    lines = ['open-meteo Wind Forecast (kts, every 3 hrs)\n']
    for i, t in enumerate(data['time'][:120:3]):
        ws  = data['wind_speed_10m'][i*3]
        wd  = data['wind_direction_10m'][i*3]
        wg  = data['wind_gusts_10m'][i*3]
        lines.append(f'{t}  {ws}kts from {wd}deg  gusts:{wg}kts')
    return '\n'.join(lines)

In [6]:
async def scrape_sources(sources, char_limit=4_000):
    results = {}
    if not sources:
        return results
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        for name, url in sources.items():
            print(f'  Scraping: {name}...')
            try:
                await page.goto(url, wait_until='domcontentloaded', timeout=30_000)
                await page.wait_for_timeout(1500)
                for sel in ['button:has-text("AGREE")', 'button:has-text("Accept")',
                            'button:has-text("Accept all")', 'button:has-text("I agree")']:
                    try:
                        await page.click(sel, timeout=1500)
                        await page.wait_for_timeout(600)
                        break
                    except:
                        pass
                await page.evaluate("""
                    document.querySelectorAll('script,style,nav,footer,header,img,input,iframe')
                             .forEach(el => el.remove())
                """)
                title = await page.title()
                body  = await page.inner_text('body')
                results[name] = (title + '\n\n' + body)[:char_limit]
                print(f'    -> {len(results[name]):,} chars')
            except Exception as e:
                results[name] = f'[Could not fetch {url}: {e}]'
        await browser.close()
    return results

## System Prompts by Category

Each category gets a tailored system prompt so the LLM knows what to focus on.

In [7]:
SYSTEM_PROMPTS = {
    'surf': """
You are an expert surf forecaster. Synthesise the raw marine, wind, and surf report data
into a clear surfer's briefing in markdown with these sections:
1. **Overview** - one paragraph verdict (is it worth going?)
2. **Swell** - height, period, direction, consistency
3. **Wind** - speed, direction, onshore/offshore assessment
4. **Best Windows** - specific days/times that look best in the 5-day window
5. **Outlook** - trend beyond 5 days if data allows
Be direct and practical. Respond in markdown, no code blocks.
""",
    'mountain': """
You are an expert mountain weather analyst. Synthesise the raw data into a hillwalker's
briefing in markdown with sections: Overview, Snow & Ice, Visibility, Wind & Temperature,
Best Windows, Outlook. Be safety-conscious. Respond in markdown, no code blocks.
""",
    'general': """
You are a weather analyst. Synthesise the raw data into a clear travel weather briefing
in markdown with sections: Overview, Rain, Wind, Temperature, Best Windows, Outlook.
Respond in markdown, no code blocks.
""",
}

## Fetch & Synthesise All Places

For each place: pulls marine + wind data from open-meteo, optionally scrapes extra sources, then runs the LLM.

In [8]:
async def run_all():
    for place_name, cfg in PLACES.items():
        print(f'\n=== {place_name} ===')
        raw = {}

        # open-meteo marine
        print('  Fetching marine data...')
        raw['Marine (open-meteo)'] = fetch_marine(cfg['lat'], cfg['lon'])

        # open-meteo wind
        print('  Fetching wind data...')
        raw['Wind (open-meteo)'] = fetch_wind(cfg['lat'], cfg['lon'])

        # scraped sources
        scraped = await scrape_sources(cfg.get('scrape_sources', {}))
        raw.update(scraped)

        # build user prompt
        user_prompt = f'Please synthesise the following forecast data for {place_name} into a structured briefing.\n\n'
        for src, content in raw.items():
            user_prompt += f'---\n### Source: {src}\n{content}\n\n'

        # call llama3.2
        print('  Synthesising with llama3.2...')
        response = ollama.chat.completions.create(
            model='llama3.2',
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPTS[cfg['category']]},
                {'role': 'user', 'content': user_prompt},
            ]
        )
        briefing = response.choices[0].message.content

        display(Markdown(f'---\n# {place_name}\n'))
        display(Markdown(briefing))

await run_all()


=== Taghazout — Surf ===
  Fetching marine data...
  Fetching wind data...
  Scraping: Anchor Point (surf-forecast.com)...
    -> 243 chars
  Synthesising with llama3.2...


---
# Taghazout — Surf


**Taghazout Surf Forecast Briefing**
=====================================

**Overview**
------------

A strong low-pressure system is dominating the North African weather pattern, bringing a robust swell to Taghazout. The surf forecast is looking very promising, with consistent and peaty waves expected throughout the week.

**Swell**
--------

* **Height:** 1.0m - 1.8m
* **Period:** 10s - 12s
* **Direction:** NNE - N
* **Consistency:** Very consistent, with waves holding well throughout the day

**Wind**
------

* **Speed:** 1.0kts - 5.0kts
* **Direction:** NW - SE
* **Onshore/offshore:** Generally offshore, with some onshore influences on Thursday and Friday

**Best Windows**
----------------

* **Monday:** 15:00 - 19:00 (stronger winds, more consistent waves)
* **Tuesday:** 12:00 - 16:00 (lighter winds, more glassy conditions)
* **Wednesday:** 15:00 - 19:00 (similar to Monday, with stronger winds and more consistent waves)
* **Thursday:** 15:00 - 19:00 (similar to Wednesday, with some onshore influences)
* **Friday:** 12:00 - 16:00 (lighter winds, more glassy conditions)
* **Saturday:** 12:00 - 16:00 (lighter winds, more glassy conditions)
* **Sunday:** 15:00 - 19:00 (stronger winds, more consistent waves)

**Outlook**
------------

The low-pressure system is expected to move northwards, bringing a decrease in swell size and period. However, the surf is likely to remain consistent and peaty, making it a great opportunity for surfers.